# dr_evt 03: the gRPC client/server, platforms as sessions

notebook 02 drove one cluster in process. This session drives clusters that live in
another process, through dr_evt's gRPC service. This is the deployment the README describes
("digital-twin controllers open independent sessions to any number of server processes"),
it is how the containers work, and until the per-job record PR lands it is the only way to
get per-job start and end times back from Python.

The service is one bidirectional streaming RPC, `SimulationService.Session`
(`src/proto/dr_evt_service.proto`). Each message from the client is one request
(`Init`, `AppendJob(s)`, `AdvanceTo`, the queries, `FinishSimulation`); the server answers
each with one response carrying the same `request_id`. One stream = one server-side
`Simulation`, on its own thread, with all of its state local to that stream. A server
process therefore hosts any number of independent simulations at once.

There are no checked-in Python stubs. `python/grpc_multi_server.py` generates them at
runtime with `grpc_tools.protoc` into a temporary directory (`load_stubs`) and wraps one
stream in a synchronous `ServerSession` (`call(request) -> response`). We reuse both.

In [1]:
from pathlib import Path
import subprocess, socket, sys, os, time, tempfile
import pandas as pd

DR_EVT = Path.cwd().resolve()                        # run from learn/ or from the repository root
while not (DR_EVT / "CMakeLists.txt").exists() and DR_EVT != DR_EVT.parent:
    DR_EVT = DR_EVT.parent
assert (DR_EVT / "CMakeLists.txt").exists(), "run this notebook from learn/ inside a dr_evt checkout"
INSTALL = Path(os.environ.get("DR_EVT_INSTALL", DR_EVT / "install"))   # the cmake install prefix
OUT = Path.cwd() / "output"                                            # scratch space, gitignored
OUT.mkdir(exist_ok=True)
SERVER_BIN = INSTALL / "bin" / "dr_evt_server"
assert SERVER_BIN.exists(), "build with -DDR_EVT_ENABLE_PROTOBUF=ON -DDR_EVT_ENABLE_GRPC=ON first"

sys.path.insert(0, str(DR_EVT / "python"))
from grpc_multi_server import load_stubs, ServerSession, read_jobs, DEFAULT_QUEUE_INPUT
QUEUE = DEFAULT_QUEUE_INPUT      # "1" in the default numeric queue-id mode

grpc, pb, service, generated_dir = load_stubs(DR_EVT)   # generates dr_evt_service_pb2{,_grpc} into a temp dir
print("stubs generated into a temporary directory")
print("client requests: ", [f.name for f in pb.ClientMessage.DESCRIPTOR.oneofs_by_name["request"].fields])
print("server responses:", [f.name for f in pb.ServerMessage.DESCRIPTOR.oneofs_by_name["response"].fields])

stubs generated into a temporary directory
client requests:  ['init', 'initialize_trace', 'advance_to', 'run_until_exclusive', 'get_current_time', 'get_nodes_in_use', 'get_available_nodes', 'get_active_job_count', 'get_fcfs_head_shadow_time', 'get_statistics', 'get_trace_size', 'append_job', 'append_jobs', 'finish_simulation', 'get_backfill_window']
server responses: ['error', 'init', 'initialize_trace', 'advance_to', 'run_until_exclusive', 'get_current_time', 'get_nodes_in_use', 'get_available_nodes', 'get_active_job_count', 'get_fcfs_head_shadow_time', 'get_statistics', 'get_trace_size', 'append_job', 'append_jobs', 'finish_simulation', 'get_backfill_window']


## Starting a server

`dr_evt_server [address]` takes exactly one optional argument and has no other flags. Two
things about the process matter:

- **Its working directory is where every session's output lands.** Each session writes
  `<session_id>.simulated.csv`, `<session_id>.resource.csv`, `<session_id>.statistics.json`
  relative to the server's cwd. In the container that is the `/data` volume. Here we start
  it in a fresh temp directory.
- It prints one "listening" line and then nothing. When stdout is a file that line is
  buffered, so do not wait on the log; wait on the port.

In [2]:
def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

SERVER_DIR = Path(tempfile.mkdtemp(dir=OUT, prefix="server_"))
ADDRESS = f"127.0.0.1:{free_port()}"
server = subprocess.Popen([str(SERVER_BIN), ADDRESS], cwd=SERVER_DIR,
                          stdout=(SERVER_DIR / "server.log").open("w"), stderr=subprocess.STDOUT)
channel = grpc.insecure_channel(ADDRESS)
grpc.channel_ready_future(channel).result(timeout=15)
channel.close()
print(f"server pid {server.pid} listening on {ADDRESS}, outputs go to {SERVER_DIR.relative_to(Path.cwd())}")

server pid 37610 listening on 127.0.0.1:64371, outputs go to output/server_1x7qa8ht


## Opening a session: `Init`

`InitRequest` is a flat subset of `SimParams` plus a required, filename-safe
`session_name`. The server appends a unique suffix (microsecond timestamp, sequence,
nonce) and returns the resulting `session_id` and the three output file names.

Two fields deserve care:

- `infile` must be a path **the server process can read**, in the same format, because
  constructing a `Simulation` parses that file's header (notebook 02's gotcha, now across
  a process boundary). Relative paths resolve against the server's cwd, so pass an
  absolute one. We use a header-only file.
- `msec_output=True` keeps sub-second precision in the output CSV; without it every
  timestamp is truncated to whole seconds.

In [3]:
HEADER_ONLY = OUT / "header_only.csv"
HEADER_ONLY.write_text("job_submit_time,num_nodes,queue,time_limit\n")

def open_session(session_name, total_nodes):
    s = ServerSession(ADDRESS, grpc, pb, service)
    init = s.call(pb.ClientMessage(init=pb.InitRequest(
        session_name=session_name,
        total_nodes=total_nodes,
        trace_format="simple",
        timestamp_format="epoch",
        backfill_policy="easy",
        priority_policy="fcfs",
        run_time_mode="limit",
        queue_impl="circular",
        msec_output=True,
        infile=str(HEADER_ONLY),
    ))).init
    return s, init

session, init = open_session("teach03", total_nodes=100)
print("ok:", init.ok)
print("session_id:", init.session_id)
print("files:", init.simulated_trace_file, "|", init.resource_trace_file, "|", init.statistics_file)

ok: True
session_id: teach03-1789412814097561-0-963540165
files: teach03-1789412814097561-0-963540165.simulated.csv | teach03-1789412814097561-0-963540165.resource.csv | teach03-1789412814097561-0-963540165.statistics.json


## Streaming over the wire

The same four operations as in process, as messages. `AppendJobsRequest` takes a
time-sorted batch; the job data fields are exactly what `read_jobs` returns. The
"append only queues, advance to evaluate" rule is unchanged: after appending at the current
time you send `AdvanceTo` for that same time again.

Below, the whole sample trace is appended up front (all submit times are in the future),
then we advance arrival by arrival and read the backfill window. The table must match
notebook 02 exactly: it is the same engine behind a socket.

In [4]:
jobs = read_jobs(DR_EVT / "python" / "examples" / "sample_trace.csv")
resp = session.call(pb.ClientMessage(append_jobs=pb.AppendJobsRequest(
    requests=[pb.JobAppendData(**job) for job in jobs])))
print("job_idx handles:", list(resp.append_jobs.job_idx))

def advance(s, t):
    s.call(pb.ClientMessage(advance_to=pb.AdvanceToRequest(target_time=t)))

def window(s):
    return s.call(pb.ClientMessage(get_backfill_window=pb.GetBackfillWindowRequest())).get_backfill_window

def stats(s):
    return s.call(pb.ClientMessage(get_statistics=pb.GetStatisticsRequest())).get_statistics

log = []
for i, job in enumerate(jobs):
    advance(session, job["submit_time"])
    w = window(session)
    st = stats(session)
    log.append(dict(t=job["submit_time"], job=i, nodes=job["num_nodes"], limit=job["limit_time"],
                    in_use=st.nodes_in_use, free=w.available_nodes, waiting=st.jobs_waiting,
                    shadow=w.shadow_time, releases=[(r.time, r.nodes_released) for r in w.releases]))
pd.DataFrame(log)

job_idx handles: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


,t,job,nodes,limit,in_use,free,waiting,shadow,releases
0,0.0,0,20,200.0,20,80,0,-1.0,[]
1,10.0,1,30,150.0,50,50,0,-1.0,[]
2,20.0,2,15,300.0,65,35,0,-1.0,[]
3,30.0,3,40,100.0,65,35,1,160.0,"[(160.0, 30)]"
4,40.0,4,25,250.0,65,35,2,160.0,"[(160.0, 30)]"
5,50.0,5,10,80.0,75,25,2,160.0,"[(130.0, 10), (160.0, 30)]"
6,60.0,6,35,180.0,75,25,3,160.0,"[(130.0, 10), (160.0, 30)]"
7,70.0,7,60,220.0,75,25,4,160.0,"[(130.0, 10), (160.0, 30)]"
8,80.0,8,20,90.0,75,25,5,160.0,"[(130.0, 10), (160.0, 30)]"
9,90.0,9,45,160.0,75,25,6,160.0,"[(130.0, 10), (160.0, 30)]"


## Closing a simulation: `FinishSimulation`

`FinishSimulationRequest` makes the server drain everything (`advance_to(max)`), write
the three files, return the final statistics plus the file names, and release the
`Simulation`. The stream stays open: a new `InitRequest` on it starts a fresh, independent
simulation. Sending `Init` while one is active is an error.

The per-job CSV is the payoff. Rows are in job-store order, so for a session that only
ever appended (no rejections) row `i` is handle `job_idx[i]`. Assert the count.

In [5]:
fin = session.call(pb.ClientMessage(finish_simulation=pb.FinishSimulationRequest())).finish_simulation
print("final:", fin.statistics)
records = pd.read_csv(SERVER_DIR / fin.simulated_trace_file)
assert len(records) == len(jobs), "a rejected job would break positional matching"
records.insert(0, "job_idx", list(resp.append_jobs.job_idx))
records["wait"] = records.begin_time - records.job_submit_time
records["runtime"] = records.end_time - records.begin_time
records["bsld"] = ((records.end_time - records.job_submit_time) / records.runtime.clip(lower=10)).clip(lower=1.0)
records

final: jobs_submitted: 10
jobs_completed: 10
current_time: 1.7976931348623157e+308
total_nodes: 100
nodes_available: 100
utilization: 0.66518987341772151
avg_wait_time: 151
avg_turnaround_time: 324
makespan: 790



,job_idx,job_submit_time,begin_time,end_time,num_nodes,exit_status,time_limit,wait,runtime,bsld
0,0,0.0,0.0,200.0,20,0,200.0,0.0,200.0,1.000000
1,1,10.0,10.0,160.0,30,0,150.0,0.0,150.0,1.000000
2,2,20.0,20.0,320.0,15,0,300.0,0.0,300.0,1.000000
3,3,30.0,160.0,260.0,40,0,100.0,130.0,100.0,2.300000
4,4,40.0,160.0,410.0,25,0,250.0,120.0,250.0,1.480000
5,5,50.0,50.0,130.0,10,0,80.0,0.0,80.0,1.000000
6,6,60.0,260.0,440.0,35,0,180.0,200.0,180.0,2.111111
7,7,70.0,410.0,630.0,60,0,220.0,340.0,220.0,2.545455
8,8,80.0,260.0,350.0,20,0,90.0,180.0,90.0,3.000000
9,9,90.0,630.0,790.0,45,0,160.0,540.0,160.0,4.375000


In [6]:
print(sorted(p.name for p in SERVER_DIR.iterdir()))
print((SERVER_DIR / fin.statistics_file).read_text())

['server.log', 'teach03-1789412814097561-0-963540165.resource.csv', 'teach03-1789412814097561-0-963540165.simulated.csv', 'teach03-1789412814097561-0-963540165.statistics.json']
{
  "jobs_submitted": 10,
  "jobs_completed": 10,
  "jobs_running": 0,
  "jobs_waiting": 0,
  "current_time": 1.79769e+308,
  "total_nodes": 100,
  "nodes_in_use": 0,
  "nodes_available": 100,
  "utilization": 0.66519,
  "avg_wait_time": 151,
  "avg_turnaround_time": 324,
  "makespan": 790
}



## Errors come back on the stream

The server wraps every request in a try/catch and answers with an `ErrorResponse`
carrying the exception text; the RPC itself never fails. `ServerSession.call` turns that
into a `RuntimeError`. Three you will meet: a request before `Init`, a second `Init` on an
active simulation, and a bad queue name.

In [7]:
fresh = ServerSession(ADDRESS, grpc, pb, service)
attempts = {
    "AdvanceTo before Init": lambda: fresh.call(pb.ClientMessage(advance_to=pb.AdvanceToRequest(target_time=1.0))),
    "Init, then Init again": lambda: (fresh.call(pb.ClientMessage(init=pb.InitRequest(session_name="twice", total_nodes=10, infile=str(HEADER_ONLY)))),
                                      fresh.call(pb.ClientMessage(init=pb.InitRequest(session_name="twice", total_nodes=10, infile=str(HEADER_ONLY))))),
    "bad queue name": lambda: fresh.call(pb.ClientMessage(append_job=pb.AppendJobRequest(submit_time=0, num_nodes=1, queue="gpu", limit_time=10))),
}
for name, attempt in attempts.items():
    try:
        attempt()
        print(f"{name}: no error")
    except RuntimeError as e:
        print(f"{name}: RuntimeError: {e}")
fresh.close()

AdvanceTo before Init: RuntimeError: 127.0.0.1:64371: Init must be called before any other request on this session
Init, then Init again: RuntimeError: 127.0.0.1:64371: Init already called on this session
bad queue name: RuntimeError: 127.0.0.1:64371: Invalid queue id: gpu


## N sessions = N platforms, and the co-allocation gap

Open one session per platform, each with its own `total_nodes`. They share nothing. That is
exactly the shape our federation client needs, and it is also where dr_evt stops: there is
no way to reserve capacity on two simulations atomically. `python/grpc_sync_coordinator.py`
makes the consequence observable with a barrier protocol:

1. Append each system's ordinary arrivals.
2. `AdvanceTo(tc)` on every system, read the state.
3. Append one fragment of a composite job to each system at `tc`.
4. `AdvanceTo(tc)` again so the fragments are evaluated.
5. Compare `nodes_in_use` before and after: a fragment "started immediately" if the delta
   covers its request. `partial_start` = some did, some did not.

Below, `alpha` is nearly full and `beta` is empty when a two-fragment composite arrives at
t=25, so exactly one fragment starts.

In [8]:
alpha, _ = open_session("alpha", total_nodes=100)
beta, _ = open_session("beta", total_nodes=100)
alpha.call(pb.ClientMessage(append_jobs=pb.AppendJobsRequest(requests=[
    pb.JobAppendData(submit_time=0, num_nodes=90, queue=QUEUE, limit_time=1000)])))   # fills alpha

composite = {"alpha": dict(submit_time=25, num_nodes=30, queue=QUEUE, limit_time=120),
             "beta":  dict(submit_time=25, num_nodes=40, queue=QUEUE, limit_time=120)}
sessions = {"alpha": alpha, "beta": beta}
for s in sessions.values():
    advance(s, 25.0)                                   # barrier at tc
before = {name: stats(s).nodes_in_use for name, s in sessions.items()}
for name, s in sessions.items():
    s.call(pb.ClientMessage(append_job=pb.AppendJobRequest(**composite[name])))
for s in sessions.values():
    advance(s, 25.0)                                   # evaluate the same-time arrivals
after = {name: stats(s).nodes_in_use for name, s in sessions.items()}

started = {name: (after[name] - before[name]) >= composite[name]["num_nodes"] for name in sessions}
print("before:", before, "after:", after)
print("started immediately:", started, "| partial_start:", any(started.values()) and not all(started.values()))
for s in sessions.values():
    s.call(pb.ClientMessage(finish_simulation=pb.FinishSimulationRequest()))
    s.close()

before: {'alpha': 90, 'beta': 0} after: {'alpha': 90, 'beta': 40}
started immediately: {'alpha': False, 'beta': True} | partial_start: True


Our federation client's answer to this: only submit a composite decision when
every leg is predicted to start now, using the backfill window as the read-only oracle;
otherwise defer it to the next window; and report `partial_start_fraction` as a metric.

## Shutting down

Close sessions, terminate the server, delete the generated stubs. Closing a stream without
`FinishSimulation` abandons the simulation: no files are written.

In [9]:
session.close()
server.terminate()
server.wait(timeout=10)
print("server exit code:", server.returncode)
print("server log:", (SERVER_DIR / "server.log").read_text().strip())
generated_dir.cleanup()

server exit code: -15
server log: 


## What this means for our client

- Reuse `ServerSession` and `load_stubs` as they are; wrap them in a `PlatformSession`
  adapter with the same interface as the in-process one.
- Pass an absolute, server-readable `infile` (a header-only file is enough) and
  `msec_output=True`.
- Capture the file names from `InitResponse`/`FinishSimulationResponse`; never
  reconstruct them.
- One server process, one session per platform. Requests within a stream are strictly
  serial; sessions are independent threads.
- Per-job records come from the session CSV after `FinishSimulation`, positionally, until
  a per-job timing query exists in both the bindings and the proto (this branch's first
  extension).
- Everything from notebook 02 still holds: no cancel, submit times non-decreasing per
  session, `advance_to` is a one-way door, oversize jobs are dropped silently.

## Exercises

1. Where do a session's output files end up, and how does the client learn their names?
2. Why does `Init` need an `infile` at all in streaming mode, and why must it be an
   absolute path here?
3. The step-through table matched notebook 02 row for row. What does that prove, and what
   does it not prove?
4. In the composite demo, what would you have to add to dr_evt to make both fragments
   start together or neither? Is that what the gate in notebook 05 does?
5. You want to run the same session twice on one stream. What is the exact message order?